# Régimes de rotation dans le Léman : notebook interactif

Jouez avec les curseurs : chaque section recalcule les nombres sans dimension, classe le régime et trace le diagramme correspondant.

**Mode d'emploi**
- Les **paramètres du lac** (section 1) sont partagés : en les changeant, toutes les sections se mettent à jour. Le préréglage *Été / Hiver* les remplit d'un coup.
- Chaque section a ses propres curseurs (vitesse, échelle, pente, etc.).
- Les **seuils de régime sont des ordres de grandeur** (voir `notes/physique_et_diagnostics.md`). Ils sont à calibrer sur le modèle.
- La physique est dans `gyres_phys.py` (fonctions pures, testables). Ce notebook ne contient que l'interface et les graphiques.

| Section | Question |
|---|---|
| 1 | Quels sont f, Rᵢ, N, et ce que la maille résout-elle ? |
| 2 | Une structure (V_max, R_max) est-elle géostrophique ou agéostrophique, et de quelle échelle (bassin, méso, sous-méso) ? |
| 3 | Une onde (période, pente de fond) propage-t-elle, est-elle résolue, la pente est-elle critique ? |
| 4 | Un front d'upwelling est-il barocliniquement instable, à quelle échelle ? |
| 5 | Le rotationnel du vent suffit-il à créer des gyres ? |
| 6 | Un cap provoque-t-il la séparation et l'émission de tourbillons ? |
| 7 | Que devient l'eau du Rhône : panache, intrusion, plongée ? |
| 8 | Un tourbillon est-il en équilibre de vent de gradient ? Est-il instable ? |
| 9 | La grille (escalier de fond, surface libre, Ekman) est-elle adaptée ? |
| 10 | Combien de mailles faut-il pour un tourbillon ? Quelle erreur sur ζ, quelle résolution effective ? |

In [1]:
%matplotlib inline
import importlib
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display, Markdown

import gyres_phys as gp
import gyres_ui as gu
importlib.reload(gp); importlib.reload(gu)

# --- style : encre sobre, un accent bleu, grille discrète (charte dataviz) ---
INK, MUTED, GRID, BAND = "#0b0b0b", "#52514e", "#e4e3df", "#f1f0ec"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "font.size": 10,
    "text.color": INK, "axes.labelcolor": INK, "axes.edgecolor": MUTED,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "lines.linewidth": 1.8, "axes.titlesize": 11, "axes.titleweight": "bold",
})

# --- widgets ---
_style = {"description_width": "230px"}
_layout = w.Layout(width="680px")

def sl(label, lo, hi, step, val, log=False, fmt=".3g"):
    if log:
        return w.FloatLogSlider(value=val, base=10, min=np.log10(lo), max=np.log10(hi),
                                step=(np.log10(hi) - np.log10(lo)) / 200, description=label,
                                style=_style, layout=_layout, readout_format=fmt,
                                continuous_update=False)
    return w.FloatSlider(value=val, min=lo, max=hi, step=step, description=label,
                         style=_style, layout=_layout, readout_format=fmt,
                         continuous_update=False)

# paramètres partagés (noms = arguments de gp.lake)
G = dict(
    lat=sl("Latitude (°N)", 45.5, 47.5, 0.05, 46.45, fmt=".2f"),
    T_epi=sl("T épilimnion (°C)", 0, 26, 0.5, 20, fmt=".1f"),
    T_hypo=sl("T hypolimnion (°C)", 0, 12, 0.5, 6, fmt=".1f"),
    h1=sl("Épaisseur épilimnion h₁ (m)", 2, 100, 1, 10, fmt=".0f"),
    H=sl("Profondeur H (m)", 20, 310, 5, 150, fmt=".0f"),
    N_th=sl("N thermocline (s⁻¹)", 2e-3, 6e-2, None, 1e-2, log=True, fmt=".2e"),
    N_hypo=sl("N hypolimnion (s⁻¹)", 5e-4, 1e-2, None, 2.5e-3, log=True, fmt=".2e"),
    dx=sl("Δx horizontal (m)", 20, 1000, None, 100, log=True, fmt=".0f"),
    dz=sl("Δz vertical (m)", 0.25, 5, 0.25, 1, fmt=".2f"),
    n_eff=sl("Résolution effective : diamètre minimal (mailles)", 4, 12, 1, 8, fmt=".0f"),
)

season = w.ToggleButtons(options=[("Été (stratifié)", "ete"), ("Hiver (peu stratifié)", "hiver")],
                         value=None, description="Préréglage :", style={"description_width": "100px"})
def _preset(change):
    p = {"ete": dict(T_epi=20, T_hypo=6, h1=10, N_th=1e-2, N_hypo=2.5e-3),
         "hiver": dict(T_epi=7, T_hypo=5.5, h1=80, N_th=2e-3, N_hypo=1e-3)}.get(change["new"])
    if p:
        for k, v in p.items():
            G[k].value = v
season.observe(_preset, names="value")

def show(func, **local):
    """Affiche les curseurs locaux + sortie ; les curseurs partagés G relancent aussi la sortie."""
    out = gu.live_output(func, {**G, **local}, key=func.__name__)
    display(w.VBox(list(local.values())), out)

def tbl(rows):
    s = "| Grandeur | Valeur | Lecture |\n|---|---|---|\n"
    return s + "\n".join(f"| {a} | {b} | {c} |" for a, b, c in rows)

def label(ax, x, y, s, color=MUTED, **kw):
    ax.text(x, y, s, color=color, fontsize=9, **kw)

## 1. Paramètres du lac et résolution de la maille

- **f** = 2Ω sin φ : paramètre de Coriolis. Période inertielle 2π/f ≈ 16,5 h.
- **Rᵢ = c₁/f** : rayon de déformation interne, avec c₁ la vitesse du mode interne 1 (modèle à deux couches : c₁ = √(g′ h₁h₂/(h₁+h₂))). C'est l'échelle au-dessus de laquelle la rotation compte (pour un bassin de largeur B, Ke = B/Rᵢ).
- **g′** = g(ρ_hypo − ρ_épi)/ρ_hypo : gravité réduite.
- **N** : fréquence de Brunt–Väisälä (stratification). Deux valeurs à régler (thermocline, hypolimnion).
- **Résolution effective** : diamètre minimal (en mailles) des structures que le modèle simule sans les amortir. Règle empirique de 5 à 8 mailles, **réglable ici (n_eff)** et utilisée dans toutes les sections ; à déterminer pour votre modèle avec le spectre d'énergie cinétique (section 10).

In [2]:
def sec1(**g):
    lk = gp.lake(**g)
    f, Ri, dx = lk["f"], lk["Ri"], lk["dx"]
    notes = ""
    if lk["gp"] <= 0:
        notes = "\n\n⚠️ g′ ≤ 0 : colonne instable ou isotherme (g′ plancher à 1e-5 m/s²). Rᵢ n'a pas de sens."
    rows = [
        ("f", f"{f:.3e} s⁻¹", f"période inertielle {lk['Ti']:.1f} h"),
        ("ρ épilimnion / hypolimnion", f"{lk['rho_epi']:.2f} / {lk['rho_hypo']:.2f} kg/m³", ""),
        ("g′", f"{lk['gp']:.4f} m/s²", ""),
        ("c₁ (mode interne 1)", f"{lk['c']:.2f} m/s", ""),
        ("Rᵢ = c₁/f", f"{Ri/1e3:.2f} km", f"Rᵢ/Δx = {Ri/dx:.0f} mailles"),
        (f"Résolution effective ≈ {lk['n_eff']:.0f} Δx (diamètre)", f"{lk['n_eff']*dx/1e3:.2f} km", f"tourbillon de rayon ≳ {lk['n_eff']/2*dx/1e3:.2f} km"),
        ("Ke du Grand Lac (B ≈ 12 km)", f"{12e3/Ri:.1f}", gp.ke_regime(12e3 / Ri)),
        ("Ke du Petit Lac (B ≈ 3,5 km)", f"{3.5e3/Ri:.1f}", gp.ke_regime(3.5e3 / Ri)),
        ("N/f thermocline / hypolimnion", f"{lk['N_th']/f:.0f} / {lk['N_hypo']/f:.0f}", ""),
        ("Pente de grille Δz/Δx", f"{lk['dz']/dx:.4f}", "pente à laquelle il y a 1 marche par maille"),
    ]
    display(Markdown(tbl(rows) + notes))

display(season)
out1 = gu.live_output(sec1, G, key="sec1")
display(w.VBox(list(G.values())), out1)

ToggleButtons(description='Préréglage :', options=(('Été (stratifié)', 'ete'), ('Hiver (peu stratifié)', 'hive…

## 2. Plan (Ke, Ro) : régimes d'échelle d'une structure en rotation

**Conventions** (les mêmes que dans la section 8 et dans les notebooks sur les tourbillons)
- **V_max** : vitesse azimutale maximale de la structure, mesurée dans son repère (translation et écoulement de fond retirés), au niveau où elle est maximale.
- **R_max** : rayon où cette vitesse est atteinte (**le rayon, pas le diamètre**).
- **Ro = V_max/(f R_max)**. Pour un cœur en rotation solide, la vorticité au centre vaut ζ/f ≈ 2 Ro.
- **Ke = R_max/Rᵢ** (Bu = Ke⁻²) : la rotation compte si Ke ≳ 1.

**Régimes d'échelle** (définis par la dynamique, pas par une taille : dans l'océan la mésoéchelle fait 10–100 km parce que Rᵢ y vaut 30–50 km ; ici Rᵢ ≈ 3 km en été et 1–2 km en hiver, donc une même structure peut changer de régime avec la saison). Seuils indicatifs, dans `gyres_phys.py` :

| Régime | Condition | Dynamique |
|---|---|---|
| Grande échelle (bassin) | Ro < 0,3 et Ke ≥ 3 | quasi-géostrophique, barotrope |
| Mésoéchelle | Ro < 0,3 et 0,5 ≤ Ke < 3 | quasi-géostrophique barocline (Eady, Kelvin, Poincaré) |
| Sous-mésoéchelle | 0,3 ≤ Ro < 3 | agéostrophique : fronts, filaments, tourbillons côtiers |
| Petite échelle | Ro ≥ 3 | rotation secondaire, turbulence 3D |
| Petite échelle quasi-géostrophique | Ro < 0,3 et Ke < 0,5 | L ≪ Rᵢ, confiné près de la surface |

**Deux lectures, deux axes.** Le **régime d'équilibre** (axe de droite, lignes pointillées) ne dépend que de Ro : géostrophique (Ro < 0,1), cyclo-géostrophique (0,1–0,5 : le terme de courbure V²/R compte), fortement agéostrophique (0,5–1), agéostrophique (Ro > 1). Le **régime d'échelle** (zones grises) combine Ro et Ke. Leurs seuils ne coïncident pas (0,3 et 3 contre 0,1, 0,5 et 1) : ce sont des repères indicatifs, pas des frontières physiques.

**Échelle verticale** quasi-géostrophique H_v = f R_max/N : si H_v > H, la structure est colonnaire ; sinon elle est confinée en surface ou à la thermocline. **Résolution** : un tourbillon de rayon R_max est résolu si R_max ≳ (n_eff/2)Δx, soit 4Δx pour n_eff = 8 (section 10).

**Systèmes typiques** (points gris, ordres de grandeur et non mesures) : grande gyre V_max = 0,05 m/s, R_max = 6 km ; tourbillon côtier 0,15 m/s, 1 km ; filament frontal 0,05 m/s, 0,3 km (demi-largeur) ; panache du Rhône 0,1 m/s, 2 km. Changez Rᵢ avec le préréglage Été / Hiver (section 1) : les zones sont fixes dans le plan, mais Ke = R_max/Rᵢ change : les points se déplacent horizontalement et une structure peut changer de régime.

In [8]:
def sec2(V, R_km, **g):
    lk = gp.lake(**g)
    f, Ri, dx, H = lk["f"], lk["Ri"], lk["dx"], lk["H"]
    R = R_km * 1e3
    Ro, Ke = V / (f * R), R / Ri
    Hv_th, Hv_hy = gp.vertical_scale(f, R, lk["N_th"]), gp.vertical_scale(f, R, lk["N_hypo"])
    rows = [
        ("Ro = V_max/(f R_max)", f"{Ro:.3f}", f"ζ/f ≈ 2 Ro = {2*Ro:.2f} au centre (cœur en rotation solide) ; {gp.ro_regime(Ro)}"),
        ("Ke = R_max/Rᵢ  (Bu = " + f"{1/Ke**2:.2f})", f"{Ke:.2f}", gp.ke_regime(Ke)),
        ("Régime d'échelle", "", gp.scale_regime(Ke, Ro)),
        ("H_v = f R_max/N (thermocline)", f"{Hv_th:.0f} m", "colonnaire" if Hv_th > H else "confiné (surface/thermocline)"),
        ("H_v = f R_max/N (hypolimnion)", f"{Hv_hy:.0f} m", "colonnaire" if Hv_hy > H else "confiné"),
        ("R_max/Δx", f"{R/dx:.0f}", "résolu" if R >= lk["n_eff"] / 2 * dx else f"sous la résolution effective (R_max < {lk['n_eff']/2:.0f}Δx)"),
    ]
    display(Markdown(tbl(rows)))

    fig, ax = plt.subplots(figsize=(8.8, 5.4))
    x = np.logspace(np.log10(0.05), np.log10(50), 200)   # Ke
    X0, X1, Y0, Y1 = 0.05, 50.0, 0.01, 10.0
    # zones de régime (teintes neutres, bornes en trait plein)
    zones = [(X0, X1, gp.RO_SMALL, Y1, "#f4f3ef"), (X0, X1, gp.RO_SUB, gp.RO_SMALL, "#dcdbd3"),
             (X0, gp.KE_MESO, Y0, gp.RO_SUB, "#f4f3ef"), (gp.KE_MESO, gp.KE_BASIN, Y0, gp.RO_SUB, "#e9e8e2"),
             (gp.KE_BASIN, X1, Y0, gp.RO_SUB, "#f4f3ef")]
    for x0_, x1_, y0_, y1_, col in zones:
        ax.fill_between([x0_, x1_], y0_, y1_, color=col, zorder=0, lw=0)
    for yy in (gp.RO_SUB, gp.RO_SMALL):
        ax.plot([X0, X1], [yy, yy], color=MUTED, lw=0.9)
    for xx in (gp.KE_MESO, gp.KE_BASIN):
        ax.plot([xx, xx], [Y0, gp.RO_SUB], color=MUTED, lw=0.9)
    label(ax, 6.5, 4.4, "petite échelle", style="italic")
    label(ax, 6.0, 0.95, "sous-mésoéchelle", style="italic")
    label(ax, 0.056, 0.2, "petite échelle QG", style="italic")
    label(ax, 0.56, 0.2, "mésoéchelle", style="italic")
    label(ax, 3.3, 0.2, "grande échelle", style="italic")
    for Vi in (0.02, 0.05, 0.1, 0.2, 0.5):
        ax.plot(x, Vi / (f * x * Ri), color=MUTED, lw=0.8, alpha=0.6)
        xl = min(40.0, Vi / (f * 0.011 * Ri))   # abscisse où la droite sort par le bas
        label(ax, xl, Vi / (f * xl * Ri), f"{Vi} m/s", ha="right", va="bottom")
    ax.plot(x, V / (f * x * Ri), color=BLUE, lw=2.2)
    ax.axvline(lk["n_eff"] / 2 * dx / Ri, color=ORANGE, lw=1.4, ls="--")
    label(ax, lk["n_eff"] / 2 * dx / Ri * 1.04, 8.5, f"R_max = {lk['n_eff']/2:.0f}Δx\n(résolution effective)", color=ORANGE, va="top")
    for name, Vp, Rp in [("grande gyre", .05, 6e3), ("tourbillon côtier", .15, 1e3),
                         ("filament frontal", .05, 300.), ("panache Rhône", .1, 2e3)]:
        ax.plot(Rp / Ri, gp.rossby(Vp, Rp, f), "o", ms=5, color=MUTED, mfc="white")
        label(ax, Rp / Ri * 1.08, gp.rossby(Vp, Rp, f) * 1.08, name, va="bottom")
    ax.plot(Ke, Ro, "o", ms=11, color=BLUE, mec="white", mew=2, zorder=5)
    label(ax, Ke * 1.12, Ro * 1.15, f"V_max = {V:.2f} m/s, R_max = {R_km:.1f} km", color=BLUE, weight="bold")
    ax.set(xscale="log", yscale="log", xlim=(X0, X1), ylim=(Y0, Y1),
           xlabel="Ke = R_max / Rᵢ", ylabel="Ro = V_max / (f R_max)")
    sec = ax.secondary_xaxis("top", functions=(lambda v: v * Ri / 1e3, lambda L_: L_ * 1e3 / Ri))
    sec.set_xlabel("R_max (km)", color=MUTED)
    # régime d'équilibre (selon Ro seul) : lignes pointillées et axe de droite
    from matplotlib.ticker import NullLocator
    for yy in (0.1, 0.5, 1.0):
        ax.plot([X0, X1], [yy, yy], color=MUTED, lw=0.6, ls=":")
    ax2 = ax.twinx()
    ax2.set_yscale("log"); ax2.set_ylim(Y0, Y1)
    ax2.set_yticks([0.0316, 0.224, 0.707, 3.16])
    ax2.set_yticklabels(["géostrophique\n(Ro < 0,1)", "cyclo-géostrophique\n(0,1–0,5)",
                         "fortement agéostrophique\n(0,5–1)", "agéostrophique\n(Ro > 1)"], fontsize=8.5, color=MUTED)
    ax2.yaxis.set_minor_locator(NullLocator()); ax2.tick_params(length=0); ax2.grid(False)
    ax2.set_ylabel("régime d'équilibre (selon Ro)", color=MUTED)
    ax.grid(False)
    plt.show()

show(sec2, V=sl("Vitesse maximale V_max (m/s)", 0.005, 1.0, None, 0.1, log=True, fmt=".3f"),
     R_km=sl("Rayon de la vitesse maximale R_max (km)", 0.1, 15, None, 1.5, log=True, fmt=".2f"))

## 3. Ondes internes : propagation, résolution, pente critique

- **Bandes de fréquence** (ω/f) : sous-inertiel (ω < f : Kelvin, topographique, réponse forcée non propagative), quasi-inertiel (≈ f), super-inertiel (f < ω < N : Poincaré, ondes internes).
- **Pente du rayon** : tanθ = √((ω²−f²)/(N²−ω²)). Près de f la propagation est quasi horizontale ; près de N elle est quasi verticale.
- **λ_z = pente × λ_h** : longueur d'onde verticale. Elle n'est résolue que si λ_z ≳ 8Δz.
- **Pente critique** : s = s_c. Sous-critique (s < s_c) : réflexion vers l'avant. Sur-critique (s > s_c) : réflexion vers le large. Critique : amplification et mélange.
- La pente de grille Δz/Δx (tirets gris) est la limite au-dessous de laquelle les rayons sont sous-résolus verticalement.

À 46°N le forçage de 24 h est sous-inertiel et celui de 12 h est super-inertiel.

In [5]:
def sec3(T_h, lam_km, s_bot, **g):
    lk = gp.lake(**g)
    f, dx, dz = lk["f"], lk["dx"], lk["dz"]
    sg = dz / dx
    wf = lk["Ti"] / T_h
    Nth_f, Nhy_f = lk["N_th"] / f, lk["N_hypo"] / f
    sc_th, sc_hy = float(gp.ray_slope(wf, Nth_f)[0]), float(gp.ray_slope(wf, Nhy_f)[0])
    lam = lam_km * 1e3
    rows = [("ω/f = T_i/T", f"{wf:.2f}", gp.wave_band(wf, Nth_f))]
    for nm, sc in (("thermocline", sc_th), ("hypolimnion", sc_hy)):
        if np.isfinite(sc):
            lz = sc * lam
            rows.append((f"pente du rayon s_c ({nm})", f"{sc:.4f}",
                         ("sous-résolue verticalement (s_c < Δz/Δx)" if sc < sg else "résolue (s_c ≥ Δz/Δx)")))
            rows.append((f"λ_z = s_c·λ_h ({nm})", f"{lz:.1f} m",
                         f"{lz/dz:.0f} niveaux : " + ("résolue" if lz >= 8 * dz else "sous-résolue (< 8Δz)")))
            rows.append((f"pente de fond s ({nm})", f"{s_bot:.3f}", gp.slope_regime(s_bot, sc)))
        else:
            rows.append((f"s_c ({nm})", "–", "pas d'onde libre à cette fréquence"))
    display(Markdown(tbl(rows)))

    fig, ax = plt.subplots(figsize=(8.4, 4.8))
    wg = np.linspace(1.0001, 4, 800)
    ax.plot(wg, gp.ray_slope(wg, Nth_f), color=BLUE, label="thermocline")
    ax.plot(wg, gp.ray_slope(wg, Nhy_f), color=ORANGE, label="hypolimnion")
    label(ax, 3.9, float(gp.ray_slope(3.9, Nth_f)[0]) * 0.55, "thermocline", color=BLUE, ha="right")
    label(ax, 3.9, float(gp.ray_slope(3.9, Nhy_f)[0]) * 1.25, "hypolimnion", color=ORANGE, ha="right")
    ax.axhline(sg, color=MUTED, ls="--", lw=1.2)
    label(ax, 0.52, sg * 1.12, "pente de grille Δz/Δx", va="bottom")
    ax.axhline(s_bot, color=AQUA, lw=1.6)
    label(ax, 0.52, s_bot * 1.12, f"pente de fond s = {s_bot:.3f}", color=AQUA, va="bottom")
    ax.axvspan(0.5, 1, color=BAND, zorder=0)
    label(ax, 0.53, 2e-3, "sous-inertiel\n(pas d'onde libre)", style="italic")
    for Tx in (12, 8):
        ax.axvline(lk["Ti"] / Tx, color=MUTED, lw=0.7, ls=":")
        label(ax, lk["Ti"] / Tx + 0.02, 1.6e-3, f"{Tx} h")
    ax.axvline(wf, color=INK, lw=1.4)
    label(ax, wf + 0.03, 0.4, f"T = {T_h:.1f} h", color=INK, weight="bold")
    ax.set(yscale="log", xlim=(0.5, 4), ylim=(1e-3, 1), xlabel="ω / f",
           ylabel="pente du rayon tanθ = λ_z / λ_h")
    ax.legend(frameon=False, loc="lower right", bbox_to_anchor=(1.0, 0.1))
    plt.show()

show(sec3,
     T_h=sl("Période du forçage T (h)", 3, 60, None, 12, log=True, fmt=".1f"),
     lam_km=sl("Longueur d'onde horizontale λ_h (km)", 0.3, 30, None, 5, log=True, fmt=".2f"),
     s_bot=sl("Pente de fond s", 0.001, 0.5, None, 0.05, log=True, fmt=".3f"))

## 4. Front d'upwelling : instabilité barocline (Eady)

- **Front** : gradient horizontal de flottabilité M² = gαΔT/L_f. Cisaillement thermique Λ = ∂U/∂z = M²/f.
- **Ri_f = N²f²/M⁴** (Richardson du front).
  - Ri_f ≳ 10 : quasi-géostrophique (Eady).
  - 1 ≲ Ri_f ≲ 10 : transition.
  - Ri_f < 1 : agéostrophique (instabilité symétrique, instabilité de couche de mélange).
- **Eady** : échelle la plus instable λ_max ≈ 3,9 R_d, avec R_d = N H_f/f. Taux de croissance maximal σ ≈ 0,31 fΛ/N.
- **Pente des isopycnes** s_iso = M²/N² : à comparer à Δz/Δx. Au-dessus, le mélange numérique le long des isopycnes inclinées augmente.

In [6]:
def sec4(dT, Lf_km, H_f, **g):
    lk = gp.lake(**g)
    f, N, dx, dz = lk["f"], lk["N_th"], lk["dx"], lk["dz"]
    alpha = gp.thermal_expansion(lk["T_epi"])
    M2 = abs(gp.G * alpha * dT) / (Lf_km * 1e3)
    Lam = M2 / f
    Ri_f = N**2 * f**2 / M2**2
    Rd = N * H_f / f
    lam_max = gp.eady_lambda_max(Rd)
    sig = gp.eady_sigma_max(f, Lam, N)
    s_iso = M2 / N**2
    reg = ("quasi-géostrophique (Eady)" if Ri_f >= 10 else
           "transition" if Ri_f >= 1 else "agéostrophique (symétrique / couche de mélange)")
    rows = [
        ("M² = gαΔT/L_f", f"{M2:.2e} s⁻²", f"α = {alpha:.2e} K⁻¹" + (" (α < 0 sous 4 °C : |M²| utilisé)" if alpha < 0 else "")),
        ("Λ = M²/f ; ΔU sur H_f", f"{Lam:.3f} s⁻¹ ; {Lam*H_f:.2f} m/s", ""),
        ("Ri_f = N²f²/M⁴", f"{Ri_f:.1f}", reg),
        ("R_d = N H_f/f", f"{Rd/1e3:.2f} km", f"Rᵢ = {lk['Ri']/1e3:.2f} km"),
        ("λ_max ≈ 3,9 R_d", f"{lam_max/1e3:.1f} km", f"{lam_max/dx:.0f} mailles : " + ("résolu" if lam_max >= lk["n_eff"] * dx else "sous-résolu")),
        ("σ_max ≈ 0,31 fΛ/N", f"{sig*86400:.2f} j⁻¹", f"temps d'e-folding {1/sig/3600:.1f} h" if sig > 0 else ""),
        ("s_iso = M²/N²  vs  Δz/Δx", f"{s_iso:.4f} vs {dz/dx:.4f}",
         "mélange numérique probable (s_iso ≳ Δz/Δx)" if s_iso >= dz / dx else "pente d'isopycne sous une marche par maille"),
    ]
    display(Markdown(tbl(rows)))

    fig, ax = plt.subplots(figsize=(8.4, 4.4))
    lamg = np.logspace(np.log10(Rd * 0.3), np.log10(Rd * 40), 300)
    s = gp.eady_growth(2 * np.pi / lamg, N, f, H_f, Lam) * 86400
    ax.plot(lamg / 1e3, s, color=BLUE)
    ax.axvline(lam_max / 1e3, color=BLUE, lw=1, ls=":")
    label(ax, lam_max / 1e3 * 1.05, s.max() * 0.35, "λ_max", color=BLUE)
    ax.axvline(lk["n_eff"] * dx / 1e3, color=ORANGE, lw=1.4, ls="--")
    label(ax, lk["n_eff"] * dx / 1e3 * 1.05, s.max() * 0.9, f"résolution effective ≈ {lk['n_eff']:.0f}Δx", color=ORANGE)
    ax.axvline(lk["Ri"] / 1e3, color=MUTED, lw=1)
    label(ax, lk["Ri"] / 1e3 * 1.05, s.max() * 0.15, "Rᵢ")
    ax.set(xscale="log", xlabel="longueur d'onde λ (km)", ylabel="taux de croissance σ (jour⁻¹)",
           ylim=(0, max(s.max(), 1e-9) * 1.15))
    plt.show()

show(sec4,
     dT=sl("Contraste thermique du front ΔT (K)", 0.2, 10, None, 1, log=True, fmt=".2f"),
     Lf_km=sl("Largeur du front L_f (km)", 0.1, 10, None, 3, log=True, fmt=".2f"),
     H_f=sl("Profondeur de la couche frontale H_f (m)", 3, 150, None, 30, log=True, fmt=".0f"))

## 5. Rotationnel du vent

- **τ = ρ_air C_d U₁₀²**, avec un coefficient de traînée C_d de Large & Pond simplifié.
- **Rotationnel** rot τ = ∂τ_y/∂x − ∂τ_x/∂y, **pompage d'Ekman** w_E = rot τ/(ρf) (positif = aspiration, dôme de thermocline).
- Ici : un vent parallèle à la côte qui décroît vers la rive sur une distance D (zone d'abri), τ(x) = τ₀ min(1, x/D), x = distance à la rive. C'est un modèle minimal du rotationnel côtier.
- **Transport d'Ekman** M = τ/(ρf) ; **upwelling côtier** (vent parallèle à la côte) w ≈ M/Rᵢ.
- **Wedderburn** W = g′h₁²/(u*²L) : W ≲ 1 la thermocline affleure (upwelling), W ≫ 1 faible inclinaison.

In [7]:
def sec5(U10, D_km, fetch_km, **g):
    lk = gp.lake(**g)
    f, Ri = lk["f"], lk["Ri"]
    tau0 = gp.wind_stress(U10)
    curl = tau0 / (D_km * 1e3)
    wE = curl / (1000 * f) * 86400
    M = tau0 / (1000 * f)
    wc = M / Ri * 86400
    W_ = gp.wedderburn(lk["gp"], lk["h1"], tau0, fetch_km * 1e3)
    rows = [
        ("τ₀ = ρ_air C_d U²", f"{tau0:.3f} N/m²", f"C_d = {gp.drag_coefficient(U10):.2e}"),
        ("rot τ (bande côtière)", f"{curl:.2e} N/m³", f"largeur de la bande : {D_km:.1f} km"),
        ("w_E = rot τ/(ρf)", f"{wE:.1f} m/jour", "aspiration (upwelling) dans la bande" ),
        ("Transport d'Ekman M", f"{M:.2f} m²/s", ""),
        ("Upwelling côtier w ≈ M/Rᵢ", f"{wc:.1f} m/jour", f"Rᵢ = {Ri/1e3:.1f} km"),
        ("Wedderburn W", f"{W_:.2f}", "thermocline affleure (W < 1)" if W_ < 1 else "faible inclinaison (W ≥ 1)"),
        ("D vs Rᵢ", f"{D_km*1e3/Ri:.1f}", "réponse barocline (D ~ Rᵢ)" if 0.3 < D_km*1e3/Ri < 3 else
         "réponse barotrope / grande gyre (D ≫ Rᵢ)" if D_km*1e3/Ri >= 3 else "échelle sub-Rᵢ"),
    ]
    display(Markdown(tbl(rows)))

    x = np.linspace(0, 30, 400)
    tau = tau0 * np.minimum(1, x / D_km)
    fig, axs = plt.subplots(1, 2, figsize=(9.2, 3.8), constrained_layout=True)
    axs[0].plot(x, tau, color=BLUE)
    axs[0].set(xlabel="distance à la rive x (km)", ylabel="τ (N/m²)", title="Tension de vent")
    axs[0].axvline(D_km, color=ORANGE, ls="--", lw=1.2)
    label(axs[0], D_km * 1.05, tau0 * 0.1, "D", color=ORANGE)
    axs[1].plot(x, np.where(x < D_km, wE, 0.0), color=BLUE)
    axs[1].axvspan(0, D_km, color=BAND, zorder=0)
    axs[1].set(xlabel="distance à la rive x (km)", ylabel="w_E (m/jour)", title="Pompage d'Ekman",
               ylim=(0, max(wE * 1.3, 1)))
    plt.show()

show(sec5,
     U10=sl("Vent U₁₀ (m/s)", 1, 25, 0.5, 7, fmt=".1f"),
     D_km=sl("Distance de décroissance D (km)", 0.5, 30, None, 10, log=True, fmt=".1f"),
     fetch_km=sl("Fetch pour Wedderburn (km)", 5, 73, 1, 70, fmt=".0f"))

## 6. Séparation à un cap

- **Ro_cap = U/(f L_cap)**, avec L_cap l'échelle de courbure du cap. **S = c_d L_cap/H** : paramètre de friction (Signell & Geyer 1991).
  - Ro_cap ≲ 1 : la rotation contraint le flux à suivre les isobathes, pas de séparation.
  - Ro_cap ≳ 1 et S ≪ 1 : séparation instationnaire, émission de tourbillons.
  - S ≳ 1 : sillage attaché et stationnaire.
- **Période d'émission** T ≈ L_cap/(St·U), avec St ~ 0,1–0,3 (nombre de Strouhal, ordre de grandeur de corps non profilés).
- **Contrôle de la maille** : nombre de Reynolds de maille UΔx/A_h, couche limite visqueuse δ ≈ A_h/U (à comparer à Δx).

Les seuils (Ro_cap = 1, S = 1) sont indicatifs.

In [8]:
def sec6(U, Lcap_km, H_cap, cd, St, Ah, **g):
    lk = gp.lake(**g)
    f, dx = lk["f"], lk["dx"]
    Lc = Lcap_km * 1e3
    Ro_c, S = U / (f * Lc), cd * Lc / H_cap
    T_s = Lc / (St * U) / 3600
    delta, Reg = Ah / U, U * dx / Ah
    rows = [
        ("Ro_cap = U/(f L_cap)", f"{Ro_c:.2f}", ""),
        ("S = c_d L_cap/H", f"{S:.3f}", ""),
        ("Régime", "", gp.cap_regime(Ro_c, S)),
        ("Période d'émission T = L/(St U)", f"{T_s:.1f} h", f"période inertielle {lk['Ti']:.1f} h"),
        ("L_cap/Δx", f"{Lc/dx:.0f}", "cap résolu" if Lc >= lk["n_eff"] / 2 * dx else f"cap sous-résolu (rayon < {lk['n_eff']/2:.0f}Δx)"),
        ("δ = A_h/U  vs  Δx", f"{delta:.1f} m vs {dx:.0f} m",
         "couche limite latérale non résolue : la production de vorticité est contrôlée par la numérique" if delta < dx else "couche limite latérale résolue"),
        ("Reynolds de maille UΔx/A_h", f"{Reg:.1f}", ""),
    ]
    display(Markdown(tbl(rows)))

    fig, ax = plt.subplots(figsize=(7.6, 5))
    ax.axvspan(1, 1e3, color=BAND, zorder=0)
    ax.fill_between([1e-3, 1], 1, 1e2, color="#e6e5df", zorder=0)
    ax.axvline(1, color=MUTED, lw=1)
    ax.axhline(1, color=MUTED, lw=1)
    label(ax, 1.3e-3, 20, "séparation instationnaire\n(émission de tourbillons)", style="italic", va="center")
    label(ax, 1.3e-3, 0.03, "suit les isobathes", style="italic")
    label(ax, 1.5, 20, "sillage attaché", style="italic")
    ax.plot(S, Ro_c, "o", ms=11, color=BLUE, mec="white", mew=2, zorder=5)
    label(ax, S * 1.3, Ro_c * 1.2, f"Ro_cap = {Ro_c:.2f}, S = {S:.2f}", color=BLUE, weight="bold")
    ax.set(xscale="log", yscale="log", xlim=(1e-3, 1e2), ylim=(1e-2, 1e2),
           xlabel="S = c_d L_cap / H  (friction)", ylabel="Ro_cap = U / (f L_cap)")
    plt.show()

show(sec6,
     U=sl("Vitesse du courant côtier U (m/s)", 0.01, 1, None, 0.15, log=True, fmt=".3f"),
     Lcap_km=sl("Échelle du cap L_cap (km)", 0.1, 10, None, 1, log=True, fmt=".2f"),
     H_cap=sl("Profondeur près du cap H (m)", 3, 200, None, 30, log=True, fmt=".0f"),
     cd=sl("Coefficient de traînée de fond c_d", 5e-4, 1e-2, None, 2.5e-3, log=True, fmt=".1e"),
     St=sl("Strouhal St", 0.05, 0.5, 0.01, 0.2, fmt=".2f"),
     Ah=sl("Viscosité horizontale A_h (m²/s)", 0.01, 10, None, 1, log=True, fmt=".2f"))

## 7. Panache du Rhône : surface, intrusion ou plongée ?

L'eau du Rhône est froide et chargée en sédiments, donc souvent **plus dense que la surface** : elle plonge en intrusion (interflow) ou en courant de turbidité. Un bulbe de surface n'existe que si l'apport est plus léger que l'eau ambiante. Le choix se fait en comparant les densités.

- Densité : ρ_Rhône = ρ_eau(T_Rhône) + 0,62 · SSC (SSC = concentration en sédiments, en kg/m³ ; 1 mg/L = 10⁻³ kg/m³).
- **Ke = W/Rᵢ** (W = largeur de l'embouchure), **Ro₀ = U₀/(fW)**, **Fr_i = U₀/c₁ = Ro₀·Ke**. Ke ≳ 1 : virage à droite (bulbe anticyclonique, courant côtier). Ke ≪ 1 : étalement radial.

In [9]:
def sec7(T_r, SSC, W_m, U0, **g):
    lk = gp.lake(**g)
    f, Ri, c = lk["f"], lk["Ri"], lk["c"]
    rho_r = float(gp.rho_water(T_r)) + 0.62 * SSC * 1e-3
    Ke, Ro0 = W_m / Ri, U0 / (f * W_m)
    rows = [
        ("ρ_Rhône", f"{rho_r:.3f} kg/m³", f"dont sédiments : +{0.62*SSC*1e-3:.3f}"),
        ("Devenir", "", gp.plume_regime(rho_r, lk["rho_epi"], lk["rho_hypo"])),
        ("Ke = W/Rᵢ", f"{Ke:.2f}", gp.plume_kelvin_regime(Ke)),
        ("Ro₀ = U₀/(fW)", f"{Ro0:.2f}", ""),
        ("Fr_i = U₀/c₁", f"{U0/c:.2f}", "supercritique" if U0 / c > 1 else "subcritique"),
    ]
    display(Markdown(tbl(rows)))

    fig, ax = plt.subplots(figsize=(8.4, 2.9))
    pts = [("épilimnion", lk["rho_epi"], MUTED), ("hypolimnion", lk["rho_hypo"], MUTED), ("Rhône", rho_r, BLUE)]
    xs = [p[1] for p in pts]
    lo, hi = min(xs), max(xs)
    pad = max((hi - lo) * 0.25, 0.05)
    ax.axvspan(min(lk["rho_epi"], lk["rho_hypo"]), max(lk["rho_epi"], lk["rho_hypo"]), color=BAND, zorder=0)
    label(ax, (lk["rho_epi"] + lk["rho_hypo"]) / 2, 0.92, "thermocline", ha="center", style="italic")
    for i, (nm, r, col) in enumerate(pts):
        ax.plot([r, r], [0, 0.85 if nm != "Rhône" else 0.65], color=col, lw=2 if nm == "Rhône" else 1.2)
        ax.plot(r, 0.65 if nm == "Rhône" else 0.85, "o", color=col, ms=8 if nm == "Rhône" else 5)
        label(ax, r, 0.45 if nm == "Rhône" else 0.2, f"{nm}\n{r:.2f}", color=col, ha="center",
              weight="bold" if nm == "Rhône" else None,
              bbox=dict(facecolor="white", edgecolor="none", pad=1.5, alpha=0.9))
    ax.set(xlim=(lo - pad, hi + pad), ylim=(0, 1), xlabel="masse volumique (kg/m³)", yticks=[])
    ax.grid(False, axis="y")
    plt.show()

show(sec7,
     T_r=sl("T Rhône (°C)", 1, 20, 0.5, 8, fmt=".1f"),
     SSC=sl("Sédiments en suspension (mg/L)", 1, 5000, None, 100, log=True, fmt=".0f"),
     W_m=sl("Largeur de l'embouchure W (m)", 50, 2000, None, 300, log=True, fmt=".0f"),
     U0=sl("Vitesse d'entrée U₀ (m/s)", 0.05, 3, None, 0.5, log=True, fmt=".2f"))

## 8. Tourbillon : équilibre de vent de gradient et instabilité

Tourbillon gaussien V(r) = V₀ (r/R) exp[(1−(r/R)²)/2] (maximum |V₀| en r = R), signé : cyclone (V > 0) ou anticyclone (V < 0). Mêmes conventions que la section 2 : |V₀| = V_max et R = R_max.

- **Vent de gradient** : fV + V²/r = (1/ρ)∂p/∂r, donc **V_g = V + V²/(fr)**. La fraction agéostrophique \|V_g − V\|/\|V\| vaut exactement **\|Ro_c\| = \|V\|/(fr)**.
- **Anticyclone** : l'équilibre n'existe que si Ro_c ≥ −1/2 (pression bornée). Au-delà, le tourbillon n'est plus en équilibre normal.
- **Instabilité inertielle (centrifuge)** : (f + 2V/r)(f + ζ) < 0 (critère de Rayleigh). Pour un anticyclone, cela correspond à ζ/f < −1 dans le cœur.
- **Résolution** : un tourbillon de rayon R est résolu si R ≳ (n_eff/2)Δx, soit 4Δx pour n_eff = 8 (section 10).

In [10]:
def sec8(V0, polarity, R_km, **g):
    lk = gp.lake(**g)
    f, dx = lk["f"], lk["dx"]
    R = R_km * 1e3
    sgn = 1.0 if polarity == "cyclone" else -1.0
    r = np.linspace(0.02, 4, 600) * R
    V, zeta = gp.gaussian_vortex(r, sgn * V0, R)
    Vg = V + V**2 / (f * r)
    Roc = V / (f * r)
    crit = (f + 2 * V / r) * (f + zeta)
    unstable = bool((crit < 0).any())
    z0 = sgn * V0 / R * 2 * np.exp(0.5)
    ageo = np.abs(Roc).max()
    anomalous = (sgn < 0) and (Roc.min() < -0.5)
    rows = [
        ("Ro_c = V/(fr), extrême", f"{Roc[np.abs(Roc).argmax()]:.2f}", gp.ro_regime(Roc[np.abs(Roc).argmax()])),
        ("ζ(0)/f", f"{z0/f:.2f}", "cœur"),
        ("Fraction agéostrophique max", f"{ageo*100:.0f} %", "écart entre V et V_g"),
        ("Équilibre de vent de gradient", "", "Ro_c < −1/2 : au-delà de la limite d'équilibre pour un anticyclone" if anomalous else "possible"),
        ("Instabilité inertielle", "", "OUI : (f+2V/r)(f+ζ) < 0 quelque part" if unstable else "non"),
        ("R/Δx", f"{R/dx:.0f}", "résolu" if R >= lk["n_eff"] / 2 * dx else f"sous-résolu (< {lk['n_eff']/2:.0f}Δx)"),
        ("Régime d'échelle (Ke = R/Rᵢ, Ro = V₀/(fR))", f"Ke = {R/lk['Ri']:.2f}, Ro = {V0/(f*R):.2f}", gp.scale_regime(R / lk["Ri"], V0 / (f * R))),
    ]
    display(Markdown(tbl(rows)))

    fig, axs = plt.subplots(1, 2, figsize=(9.6, 4), constrained_layout=True)
    ax = axs[0]
    ax.plot(r / 1e3, V, color=BLUE)
    ax.plot(r / 1e3, Vg, color=ORANGE, ls="--")
    ax.axhline(0, color=MUTED, lw=0.6)
    i1, i2 = np.abs(V).argmax(), np.abs(Vg).argmax()
    label(ax, r[i1] / 1e3 + 0.12 * R_km, V[i1], "V (vitesse réelle)", color=BLUE, va="center")
    label(ax, r[i2] / 1e3 + 0.12 * R_km, Vg[i2], "V_g (géostrophique)", color=ORANGE, va="center")
    ax.set(xlabel="rayon r (km)", ylabel="vitesse azimutale (m/s)", title="Vent de gradient")
    ax = axs[1]
    ax.plot(r / 1e3, zeta / f, color=BLUE)
    ax.axhline(-1, color=ORANGE, ls="--", lw=1.2)
    label(ax, r[-1] / 1e3, -1.05, "f + ζ = 0", color=ORANGE, ha="right", va="top")
    ax.axhspan(min(-1.5, (zeta / f).min() * 1.05), -1, color=BAND, zorder=0)
    ax.axhline(0, color=MUTED, lw=0.6)
    ax.set(xlabel="rayon r (km)", ylabel="ζ / f", title="Vorticité")
    plt.show()

show(sec8,
     V0=sl("Vitesse maximale |V₀| (m/s)", 0.01, 0.8, None, 0.1, log=True, fmt=".3f"),
     polarity=w.ToggleButtons(options=["cyclone", "anticyclone"], value="anticyclone", description="Polarité",
                              style={"description_width": "230px"}),
     R_km=sl("Rayon du maximum R (km)", 0.2, 10, None, 2.5, log=True, fmt=".2f"))

## 9. La grille est-elle adaptée ? (escalier, surface libre, Ekman)

- **Escalier de fond** : n = s·Δx/Δz niveaux franchis par maille. Pente de grille s_g = Δz/Δx. Régimes : s ≪ s_g marches larges ; s ≈ s_g une marche par maille ; s ≫ s_g « mur ».
- **Surface libre** : ε = |η|/Δz. ε < 0,1 négligeable ; 0,1–0,5 couche du haut ±10–50 % (artefact sur température de surface) ; ≥ 1 la couche s'annule (z* ou couche épaisse).
- **Ekman** : δ_E = √(2A_v/f). Résolue si δ_E ≳ 3Δz.
- **Couche de mélange diurne** D : sous-résolue si D ≲ 3Δz.

In [11]:
def sec9(s, eta, Av, Dml, **g):
    lk = gp.lake(**g)
    f, dx, dz = lk["f"], lk["dx"], lk["dz"]
    sg = dz / dx
    n = s * dx / dz
    eps = eta / dz
    dE = np.sqrt(2 * Av / f)
    rows = [
        ("Pente de grille s_g = Δz/Δx", f"{sg:.4f}", ""),
        ("n = s Δx/Δz", f"{n:.1f} niveaux par maille", gp.grid_slope_regime(s, sg)),
        ("ε = abs(η)/Δz", f"{eps:.2f}", gp.eps_regime(eps)),
        ("δ_E = √(2A_v/f)", f"{dE:.1f} m ({dE/dz:.1f} niveaux)", "couche d'Ekman résolue" if dE >= 3 * dz else "sous-résolue (< 3Δz)"),
        ("Couche de mélange D", f"{Dml:.1f} m ({Dml/dz:.1f} niveaux)", "résolue" if Dml >= 3 * dz else "sous-résolue (< 3Δz) : température de surface et cisaillement biaisés"),
    ]
    display(Markdown(tbl(rows)))

    ncell = 12
    xs = np.arange(ncell + 1) * dx
    true = s * xs
    zlev = np.round(s * (xs[:-1] + dx / 2) / dz) * dz
    fig, ax = plt.subplots(figsize=(8.4, 3.6))
    ax.step(xs, np.append(zlev, zlev[-1]), where="post", color=ORANGE, label="fond du modèle (niveaux z)")
    ax.plot(xs, true, color=BLUE, label="fond réel")
    ax.invert_yaxis()
    ax.set(xlabel="distance horizontale (m)", ylabel="profondeur (m)")
    ax.legend(frameon=False, loc="lower left")
    plt.show()

show(sec9,
     s=sl("Pente de fond s", 0.001, 0.5, None, 0.05, log=True, fmt=".3f"),
     eta=sl("Amplitude de la seiche |η| (m)", 0.0, 1.5, 0.05, 0.3, fmt=".2f"),
     Av=sl("Viscosité verticale A_v (m²/s)", 1e-5, 1e-2, None, 1e-3, log=True, fmt=".1e"),
     Dml=sl("Couche de mélange diurne D (m)", 0.3, 20, None, 2, log=True, fmt=".1f"))

## 10. Résolution : combien de mailles pour un tourbillon ?

Trois critères différents, souvent confondus :
1. **Représenter la forme** : au minimum, diamètre ≥ 2Δx (limite de Nyquist) ; en dessous, la structure n'existe pas sur la grille.
2. **Calculer ζ à partir du champ de vitesse** : différences centrées, erreur cinématique de ζ (courbe ci-dessous). Elle est faible dès R_max ≈ 3Δx (≈ 5 %) mais ne dépend que de la grille.
3. **Maintenir la structure dans le modèle** : le schéma d'advection, la viscosité et l'hyperviscosité amortissent les structures de 2 à 4 mailles de large. La **résolution effective** est de l'ordre de 5 à 8 mailles en diamètre, soit un rayon minimal de 2,5 à 4Δx (règle empirique de la littérature, valeur à vérifier pour votre modèle).

Le paramètre **n_eff** (section 1) fixe le diamètre minimal en mailles ; il pilote tous les drapeaux « résolu / sous-résolu » du notebook. Pour le déterminer pour **votre** modèle : calculer le spectre d'énergie cinétique horizontal d'une sortie ; la résolution effective est la longueur d'onde où le spectre s'écarte de sa loi de puissance vers le bas (en général 5–8 Δx), et n_eff vaut cette longueur d'onde divisée par Δx.

Le calcul de l'erreur cinématique (tourbillon gaussien, centre sur un nœud) est fait par `gp.vortex_zeta_discretization` ; la réponse du gradient d'une onde par `gp.gradient_response`.

In [12]:
def sec10(R_km, **g):
    lk = gp.lake(**g)
    dx, ne, Ri = lk["dx"], lk["n_eff"], lk["Ri"]
    R = R_km * 1e3
    r_dx = R / dx
    peak, err = gp.vortex_zeta_discretization(r_dx)
    r_min = ne / 2 * dx
    if r_dx < 1:
        status = "sous la maille : non représentable"
    elif r_dx < 2:
        status = "limite de Nyquist (diamètre < 4Δx) : ζ très sous-estimé, forme déformée"
    elif r_dx < ne / 2:
        status = f"représentable, mais amorti par le schéma (sous la résolution effective de {ne:.0f}Δx de diamètre)"
    else:
        status = "résolu"
    resp = float(gp.gradient_response(2 * r_dx))
    rows = [
        ("R_max/Δx ; diamètre/Δx", f"{r_dx:.1f} ; {2*r_dx:.1f}", status),
        ("Erreur cinématique de ζ au centre", f"sous-estimé de {100*(1-peak):.1f} % (erreur max du profil {100*err:.1f} %)",
         "différences centrées sur le champ exact ; n'inclut pas l'amortissement par le schéma" + (" (asymptote 0,5/R² pour R > 30Δx)" if r_dx > 30 else "")),
        ("Réponse du gradient à λ = diamètre", f"{100*resp:.0f} %", "pente d'une onde sin(kx) par différence centrée"),
        (f"Rayon minimal résolu (diamètre ≥ {ne:.0f} mailles)", f"{r_min/1e3:.2f} km", f"Ke_min = R_min/Rᵢ = {r_min/Ri:.2f}"),
    ]
    display(Markdown(tbl(rows)))

    rg = np.geomspace(0.5, 30, 40)
    errs = np.array([100 * (1 - gp.vortex_zeta_discretization(r_)[0]) for r_ in rg])
    lg = np.geomspace(2.05, 40, 200)
    fig, axs = plt.subplots(1, 2, figsize=(10.4, 4.2), constrained_layout=True)
    ax = axs[0]
    ax.loglog(rg, errs, color=BLUE)
    ax.axvline(ne / 2, color=ORANGE, lw=1.4, ls="--"); ax.text(ne / 2 * 0.93, 0.12, f"résolution effective\n(R = {ne/2:.0f}Δx)", color=ORANGE, fontsize=9, ha="right")
    ax.axvline(2, color=MUTED, lw=1, ls=":"); ax.text(2 * 1.05, 40, "Nyquist\n(diamètre 4Δx)", color=MUTED, fontsize=9, va="top")
    if 0.5 <= r_dx <= 30:
        ax.plot(r_dx, 100 * (1 - peak), "o", ms=10, color=BLUE, mec="white", mew=2, zorder=5)
        ax.annotate(f"R_max = {r_dx:.1f}Δx : {100*(1-peak):.1f} %", (r_dx, 100 * (1 - peak)), xytext=(8, 8), textcoords="offset points", color=BLUE, fontsize=9.5, weight="bold")
    ax.set(xlabel="R_max / Δx", ylabel="sous-estimation de ζ au centre (%)", title="Erreur cinématique de ζ (différences centrées)", ylim=(0.05, 60))
    ax = axs[1]
    ax.semilogx(lg, 100 * gp.gradient_response(lg), color=BLUE)
    ax.axhline(100, color=MUTED, lw=0.6)
    ax.axvline(ne, color=ORANGE, lw=1.4, ls="--"); ax.text(ne * 1.05, 20, f"résolution effective\n(λ = {ne:.0f}Δx)", color=ORANGE, fontsize=9)
    if 2.05 <= 2 * r_dx <= 40:
        ax.plot(2 * r_dx, 100 * resp, "o", ms=10, color=BLUE, mec="white", mew=2, zorder=5)
        ax.annotate(f"λ = diamètre = {2*r_dx:.1f}Δx : {100*resp:.0f} %", (2 * r_dx, 100 * resp), xytext=(8, -16), textcoords="offset points", color=BLUE, fontsize=9.5, weight="bold")
    ax.set(xlabel="longueur d'onde λ / Δx", ylabel="pente numérique / pente exacte (%)", title="Gradient d'une onde par différence centrée", ylim=(0, 105))
    plt.show()

show(sec10, R_km=sl("Rayon de la vitesse maximale R_max (km)", 0.05, 15, None, 1.5, log=True, fmt=".2f"))

---
**Suite** : appliquer ces régimes aux sorties du modèle (ζ/f, Okubo–Weiss, fraction agéostrophique, budget de vorticité, suivi de tourbillons). Voir `CLAUDE.md` pour la liste des tâches et `notes/physique_et_diagnostics.md` pour les définitions.